# Build Local Knowledge Index (Experimental Phase 1)

This notebook builds a **local-only** knowledge index from files in `agent/ui/knowledge/`.

What this does:
- Loads `.txt`, `.md`, and `.pdf` files
- Splits documents into overlapping chunks
- Builds a BM25 keyword index
- Saves index artifacts in `agent/ui/storage/`

## Step 1: Confirm Local-Only Setup and Paths

Before indexing, this notebook resolves the project root and ensures the two working folders exist:
- `agent/ui/knowledge/` for source documents
- `agent/ui/storage/` for index artifacts

Expected input:
- A valid project checkout containing `AGENTS.md` and `agent/`.

Expected output/side effects:
- Prints resolved project and folder paths.
- Creates `knowledge/` and `storage/` folders if they do not already exist.

Privacy note: indexing and retrieval stay local; no files are uploaded during these steps.

In [6]:
from pathlib import Path
import sys

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    for candidate in candidates:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'agent').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate project root containing AGENTS.md and agent/.')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

KNOWLEDGE_DIR = PROJECT_ROOT / 'agent' / 'ui' / 'knowledge'
STORAGE_DIR = PROJECT_ROOT / 'agent' / 'ui' / 'storage'

KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Knowledge dir: {KNOWLEDGE_DIR}')
print(f'Storage dir: {STORAGE_DIR}')

Project root: /Users/cjcscha/ROBERT/helper_rob/robert
Knowledge dir: /Users/cjcscha/ROBERT/helper_rob/robert/agent/ui/knowledge
Storage dir: /Users/cjcscha/ROBERT/helper_rob/robert/agent/ui/storage


## Step 2: Add Documents to the Knowledge Folder

Put your local reference files into `agent/ui/knowledge/` before scanning.

Expected input:
- One or more `.txt`, `.md`, or `.pdf` files in `agent/ui/knowledge/` (subfolders are allowed).

Expected output/side effect:
- No files are changed yet.
- The next scan cell will list only supported files and ignore unsupported extensions.

In [7]:
from agent.ui.rag.loaders import scan_knowledge_files

files = scan_knowledge_files(KNOWLEDGE_DIR)
print(f'Found {len(files)} supported file(s):')
for path in files:
    print(' -', path.relative_to(PROJECT_ROOT))

if not files:
    print('No files found yet. Add documents to agent/ui/knowledge/ and rerun this cell.')

Found 6 supported file(s):
 - agent/ui/knowledge/ROBERT - Bridging the gap - WIREs Comput Mol Sci - 2024 - Dalmau - ROBERT  Bridging the Gap Between Machine Learning and Chemistry (1).pdf
 - agent/ui/knowledge/ROBERT Supporting Information.pdf
 - agent/ui/knowledge/ROBERT_DOCS_combined_1.txt
 - agent/ui/knowledge/ROBERT_DOCS_combined_2.txt
 - agent/ui/knowledge/ROBERT_DOCS_combined_3.txt
 - agent/ui/knowledge/WIREs Comput Mol Sci - 2023 - Alegre‐Requena - AQME  Automated quantum mechanical environments for researchers and.pdf


## Step 3: Build the Local BM25 Index

This step runs the reusable `build_local_index()` function from `agent/ui/rag/build_index.py`.

Expected input:
- Files already present in `agent/ui/knowledge/`.
- Chunking parameters (`chunk_size`, `chunk_overlap`).

Expected output/side effects:
- Creates/updates `agent/ui/storage/chunks.jsonl` (all text chunks with metadata).
- Creates/updates `agent/ui/storage/bm25.pkl` (serialized BM25 index).
- Creates/updates `agent/ui/storage/manifest.json` (index build metadata).
- Prints a summary with files indexed and chunks created.

In [8]:
from agent.ui.rag.build_index import build_local_index

summary = build_local_index(
    knowledge_dir=KNOWLEDGE_DIR,
    storage_dir=STORAGE_DIR,
    chunk_size=350,
    chunk_overlap=75,
)

print('Index build complete.')
print(f"Files indexed: {summary['num_files_indexed']}")
print(f"Chunks created: {summary['num_chunks_created']}")
print(f"chunks.jsonl: {summary['chunks_path']}")
print(f"bm25.pkl: {summary['bm25_path']}")
print(f"manifest.json: {summary['manifest_path']}")

Index build complete.
Files indexed: 6
Chunks created: 198
chunks.jsonl: /Users/cjcscha/ROBERT/helper_rob/robert/agent/ui/storage/chunks.jsonl
bm25.pkl: /Users/cjcscha/ROBERT/helper_rob/robert/agent/ui/storage/bm25.pkl
manifest.json: /Users/cjcscha/ROBERT/helper_rob/robert/agent/ui/storage/manifest.json


## Step 4: Test Retrieval Quality

Use a representative question and inspect the top retrieved chunks.

Expected input:
- A query string in the code cell below.
- Existing index artifacts in `agent/ui/storage/` from Step 3.

Expected output:
- Top-k retrieved chunks with source file, chunk index, and BM25 score.
- A formatted context block you can later pass into chat prompting.

If retrieval looks poor after adding/editing documents, rebuild the index first (Step 3).

In [9]:
from agent.ui.rag.retrieve import retrieve, build_context

query = 'Why did I get this score?'
k = 5

results = retrieve(query=query, k=k, storage_dir=STORAGE_DIR)
print(f'Query: {query}')
print(f'Retrieved: {len(results)} chunk(s)')

for i, item in enumerate(results, start=1):
    preview = item['text'][:220].replace('\n', ' ')
    print(f"\n[{i}] {item['source_name']} | chunk {item['chunk_index']} | score={item['score']:.4f}")
    print(preview + ('...' if len(item['text']) > 220 else ''))

context_block = build_context(results)
print('\nFormatted context preview:')
print(context_block[:1200] + ('...' if len(context_block) > 1200 else ''))

Query: Why did I get this score?
Retrieved: 5 chunk(s)

[1] ROBERT_DOCS_combined_1.txt | chunk 13 | score=10.3273
[`report.print_score()`](#robert.report.report.print_score) * [`report.print_warnings()`](#robert.report.report.print_warnings) * [`report.print_y_distrib()`](#robert.report.report.print_y_distrib) * [`report.transpa_mod...

[2] ROBERT_DOCS_combined_1.txt | chunk 17 | score=7.7360
the y values, based on R\*\*2 values c) reduces the number of descriptors to one third of the datapoints using RFECV. create\_heatmap(*self*, *csv\_df*, *suffix*, *path\_raw*)[](#robert.utils.create_heatmap "Link to thi...

[3] ROBERT_DOCS_combined_3.txt | chunk 23 | score=7.1755
--- | --- | | * [analyze\_tests() (verify method)](API/robert.verify.html#robert.verify.verify.analyze_tests) | * [analyze\_warnings() (report method)](API/robert.report.html#robert.report.report.analyze_warnings) * [aqm...

[4] ROBERT Supporting Information.pdf | chunk 2 | score=6.7721
scikit-learn-intelex Intel® Extens

## Step 5: Compare Chunking Settings (Optional but Recommended)

This experiment loops over multiple `(chunk_size, chunk_overlap)` settings and runs retrieval on a small query set.

Expected input:
- A short list of candidate settings.
- A few realistic questions you expect users to ask.

Expected output:
- Side-by-side retrieval previews for each setting.
- A practical choice of chunking defaults for your document set.

Side effect:
- Each loop rebuilds the local index in `agent/ui/storage/` with the current setting.

In [5]:
from agent.ui.rag.build_index import build_local_index
from agent.ui.rag.retrieve import retrieve

SETTINGS = [
    (350, 75),
    (500, 100),
    (700, 120),
]

EVAL_QUERIES = [
    'Why did I get this score?',
    'What does CV versus test gap mean?',
    'How can I improve descriptor relevance?',
]

for chunk_size, chunk_overlap in SETTINGS:
    print('=' * 80)
    print(f'Setting: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}')

    build_local_index(
        knowledge_dir=KNOWLEDGE_DIR,
        storage_dir=STORAGE_DIR,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    for query in EVAL_QUERIES:
        results = retrieve(query=query, k=2, storage_dir=STORAGE_DIR)
        print(f"\nQuery: {query}")
        for i, item in enumerate(results, start=1):
            preview = item['text'][:180].replace('\n', ' ')
            print(
                f"  [{i}] {item['source_name']} | chunk {item['chunk_index']} | score={item['score']:.4f}"
            )
            print(f"      {preview}{'...' if len(item['text']) > 180 else ''}")

print('\nDone. Pick the setting with the most focused, relevant chunk previews for your questions.')

Setting: chunk_size=350, chunk_overlap=75

Query: Why did I get this score?
  [1] ROBERT_DOCS_combined_1.txt | chunk 13 | score=10.3273
      [`report.print_score()`](#robert.report.report.print_score) * [`report.print_warnings()`](#robert.report.report.print_warnings) * [`report.print_y_distrib()`](#robert.report.report...
  [2] ROBERT_DOCS_combined_1.txt | chunk 17 | score=7.7360
      the y values, based on R\*\*2 values c) reduces the number of descriptors to one third of the datapoints using RFECV. create\_heatmap(*self*, *csv\_df*, *suffix*, *path\_raw*)[](#...

Query: What does CV versus test gap mean?
  [1] ROBERT_DOCS_combined_2.txt | chunk 12 | score=6.5798
      examples presented in the ROBERT publication (DOI: <https://doi.org/10.1002/wcms.1733>) along with eight additional examples from low-data regimes (DOI: <https://doi.org/10.1039/D5...
  [2] ROBERT_DOCS_combined_3.txt | chunk 27 | score=6.1985
      hyperoptimized models using > multiple cross-validation techniques. 

## How to Interpret Evaluation Output

You just compared multiple chunk settings with your own questions.

Use this rule of thumb:
1. Prefer settings where top chunks are focused and directly answer the question.
2. If top chunks are too broad, use smaller chunks.
3. If answers miss needed context, try slightly larger chunks.

Recommended starting default for this project: `chunk_size=350`, `chunk_overlap=75`.

## Step 6: Use the Index in the UI Workflow

At this point the local knowledge index exists on disk and retrieval is available.

What to do next:
1. Start the ROBERT UI app as usual.
2. Keep your knowledge files in `agent/ui/knowledge/` under version control policy you prefer.
3. Re-run Step 3 whenever files are added, removed, or edited.

Important behavior:
- Indexing and retrieval are local-only.
- Chat integration uses retrieval artifacts in a later workflow step.
- Retrieval quality depends on both document content and chunking settings.